In [28]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
import os
load_dotenv()

True

In [29]:
llm = ChatOpenAI()

In [30]:
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
import requests
from langgraph.prebuilt import ToolNode,tools_condition

In [31]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [32]:
client = MultiServerMCPClient(
    {
        "arith": {
            "transport": "stdio",
            "command": "python3",          
            "args": ["/Users/nitish/Desktop/mcp-math-server/main.py"],
        },
        "expense": {
            "transport": "streamable_http",  # if this fails, try "sse"
            "url": "https://splendid-gold-dingo.fastmcp.app/mcp"
        }
    }
)

In [33]:
# Tools 
search_tools = DuckDuckGoSearchRun()

@tool
def calculator(first_num: float,second_num: float,operation: str)->dict:
    """Perform a basic arithmetic operation betweeen two numbers .
    Supported add,sub,mul,div"""

    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}
        
        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {'error':str(e)}

@tool
def get_stock_price(symbol:str)->dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA') 
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=C9PE94QUEW9VWGFM"

    r = requests.get(url=url)
    return r.json()

In [34]:
# state
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [35]:
async def build_graph():
    tools = client.get_tools()
    print(tools)
    # bind llm with tools
    llm_with_tools = llm.bind_tools(tools=[search_tools,calculator,get_stock_price])

        # graph nodes
    async def chat_node(state: ChatState):
        """LLM node that may answer or request a tool call."""
        messages = state['messages']
        response = await llm_with_tools.ainvoke(messages)
        return {"messages": [response]}
    
        # graph structure
    graph = StateGraph(ChatState)
    graph.add_node("chat_node", chat_node)
    graph.add_node("tools", ToolNode(tools=[search_tools, calculator, get_stock_price]))

    graph.add_edge(START, "chat_node")
    # If the LLM asked for a tool, go to ToolNode; else finish
    graph.add_conditional_edges('chat_node',tools_condition)
    graph.add_edge('tools','chat_node')

    chatbot = graph.compile()

    return chatbot
    

In [49]:
async def main():
    chatbot = await build_graph()

    result = await chatbot.ainvoke(input={'messages':[HumanMessage(content='what is my expense till now with month ')]})

    print(result['messages'][-1].content)
await main()

<coroutine object MultiServerMCPClient.get_tools at 0x00000145A21209E0>


C:\Users\rajen\AppData\Local\Temp\ipykernel_8468\3940568020.py:2: RuntimeWarning: coroutine 'MultiServerMCPClient.get_tools' was never awaited
  chatbot = await build_graph()


I can help you calculate your total expenses for the current month. Please provide me with the necessary information to proceed.
